In [36]:
import os
from pathlib import Path
from typing import List, Optional, Union

import numpy as np
import pandas as pd


def normalize_csvs_to_tri_ctrl_median(
    input_folder: Union[str, os.PathLike],
    *,
    treatment_col: str = "Treatment",
    value_col: str = "Results",
    ctrl_label: str = "6hr_Ctrl",
    output_suffix: str = "_Norm",
    file_glob: str = "*.csv",
    recursive: bool = False,
    norm_col: Optional[str] = None,
) -> List[Path]:
    """
    For each CSV in a folder:
      - compute median(value_col) within rows where Treatment == ctrl_label
      - write normalized values to a new column (default: f"{value_col}_Norm")
      - save new CSV as <original>_Norm.csv next to original (no overwrite)

    Note: does NOT save the median used as a column.
    """
    input_folder = Path(input_folder)
    if not input_folder.exists() or not input_folder.is_dir():
        raise ValueError(f"input_folder is not a directory: {input_folder}")

    pattern = f"**/{file_glob}" if recursive else file_glob
    csv_paths = sorted(input_folder.glob(pattern))
    if not csv_paths:
        raise ValueError(f"No CSV files found in: {input_folder} (glob='{file_glob}', recursive={recursive})")

    if norm_col is None:
        norm_col = f"{value_col}_Norm"

    out_paths: List[Path] = []

    for p in csv_paths:
        df = pd.read_csv(p)

        if treatment_col not in df.columns:
            raise ValueError(f"[{p.name}] Missing required column '{treatment_col}'")
        if value_col not in df.columns:
            raise ValueError(f"[{p.name}] Missing required column '{value_col}'")

        vals = pd.to_numeric(df[value_col], errors="coerce")

        ctrl_mask = df[treatment_col].astype(str).str.strip() == ctrl_label
        ctrl_vals = vals[ctrl_mask].dropna()

        if ctrl_vals.empty:
            raise ValueError(f"[{p.name}] No valid numeric '{value_col}' values for control group '{ctrl_label}'")

        ctrl_median = float(np.median(ctrl_vals))
        if not np.isfinite(ctrl_median) or ctrl_median == 0:
            raise ValueError(f"[{p.name}] Control median is invalid (median={ctrl_median})")

        df[norm_col] = vals / ctrl_median

        out_path = p.with_name(p.stem + output_suffix + p.suffix)

        # avoid overwriting
        if out_path.exists():
            i = 1
            while True:
                candidate = p.with_name(f"{p.stem}{output_suffix} ({i}){p.suffix}")
                if not candidate.exists():
                    out_path = candidate
                    break
                i += 1

        df.to_csv(out_path, index=False)
        out_paths.append(out_path)
        print(f"✓ {p.name} -> {out_path.name}")

    return out_paths


normalize_csvs_to_tri_ctrl_median(
    '/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/6hr'
)

✓ 6hr_Area (µm²).csv -> 6hr_Area (µm²)_Norm.csv
✓ 6hr_Aspect Ratio.csv -> 6hr_Aspect Ratio_Norm.csv
✓ 6hr_Circularity.csv -> 6hr_Circularity_Norm.csv
✓ 6hr_Convex Hull Area (µm²).csv -> 6hr_Convex Hull Area (µm²)_Norm.csv
✓ 6hr_Convex Hull Perimeter (µm).csv -> 6hr_Convex Hull Perimeter (µm)_Norm.csv
✓ 6hr_Perimeter (µm).csv -> 6hr_Perimeter (µm)_Norm.csv
✓ 6hr_Roughness.csv -> 6hr_Roughness_Norm.csv


[PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/6hr/6hr_Area (µm²)_Norm.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/6hr/6hr_Aspect Ratio_Norm.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/6hr/6hr_Circularity_Norm.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/6hr/6hr_Convex Hull Area (µm²)_Norm.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/6hr/6hr_Convex Hull Perimeter (µm)_Norm.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/6hr/6hr_Perimeter (µm)_Norm.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/6hr/6hr_Roughness_Norm.csv')]

UPDATED TO LOG BASE 2 FOLD CHANGE

In [43]:
import os
from pathlib import Path
from typing import List, Optional, Union

import numpy as np
import pandas as pd


def normalize_csvs_to_ctrl_median_whole_file(
    input_folder: Union[str, os.PathLike],
    *,
    value_col: str = "Results",
    filename_col: Union[str, int] = 0,      # first column contains the "file name" / key
    ctrl_token: str = "6hr_Ctrl",           # rows whose key contains this token are controls
    output_suffix: str = "_Norm",
    file_glob: str = "*.csv",
    recursive: bool = False,
    norm_col: Optional[str] = None,
    log2fc_col: Optional[str] = None,
    pseudocount: float = 0.0,
    skip_already_normalized: bool = True,
) -> List[Path]:
    """
    For each CSV in input_folder:
      - find control rows where the *first column* contains ctrl_token (e.g., "6hr_Ctrl")
      - compute ctrl_median = median(value_col) across those control rows (within that CSV)
      - normalize ALL rows in the CSV by that ctrl_median
      - write:
          * <value_col>_Norm   = (value+pseudocount)/(ctrl_median+pseudocount)
          * <value_col>_log2FC = log2(<value_col>_Norm)
      - save as <original>_Norm.csv (no overwrite)

    This ignores prefix grouping entirely (since you want the single 6hr_Ctrl median applied
    to all rows in the file).
    """
    input_folder = Path(input_folder)
    if not input_folder.exists() or not input_folder.is_dir():
        raise ValueError(f"input_folder is not a directory: {input_folder}")

    pattern = f"**/{file_glob}" if recursive else file_glob
    csv_paths = sorted(input_folder.glob(pattern))
    if not csv_paths:
        raise ValueError(f"No CSV files found in: {input_folder} (glob='{file_glob}', recursive={recursive})")

    if norm_col is None:
        norm_col = f"{value_col}_Norm"
    if log2fc_col is None:
        log2fc_col = f"{value_col}_log2FC"

    out_paths: List[Path] = []

    for p in csv_paths:
        if skip_already_normalized and p.stem.endswith(output_suffix):
            continue

        df = pd.read_csv(p)

        # Resolve key column (first column by default)
        if isinstance(filename_col, int):
            if filename_col < 0 or filename_col >= df.shape[1]:
                raise ValueError(f"[{p.name}] filename_col index {filename_col} out of range (ncol={df.shape[1]})")
            key_series = df.iloc[:, filename_col]
            key_col_label = df.columns[filename_col]
        else:
            if filename_col not in df.columns:
                raise ValueError(f"[{p.name}] Missing filename_col '{filename_col}'")
            key_series = df[filename_col]
            key_col_label = filename_col

        if value_col not in df.columns:
            raise ValueError(f"[{p.name}] Missing required column '{value_col}'")

        keys = key_series.astype(str).str.strip()
        vals = pd.to_numeric(df[value_col], errors="coerce")

        ctrl_mask = keys.str.contains(ctrl_token, regex=False)
        ctrl_vals = vals[ctrl_mask].dropna()

        if ctrl_vals.empty:
            raise ValueError(
                f"[{p.name}] No valid numeric '{value_col}' values found for control rows "
                f"(where '{key_col_label}' contains '{ctrl_token}')."
            )

        ctrl_median = float(np.median(ctrl_vals))
        if not np.isfinite(ctrl_median) or ctrl_median == 0:
            raise ValueError(f"[{p.name}] Control median is invalid (median={ctrl_median})")

        fc = (vals + pseudocount) / (ctrl_median + pseudocount)
        df[norm_col] = fc
        df[log2fc_col] = np.log2(fc)

        out_path = p.with_name(p.stem + output_suffix + p.suffix)
        if out_path.exists():
            i = 1
            while True:
                candidate = p.with_name(f"{p.stem}{output_suffix} ({i}){p.suffix}")
                if not candidate.exists():
                    out_path = candidate
                    break
                i += 1

        df.to_csv(out_path, index=False)
        out_paths.append(out_path)
        print(f"✓ {p.name} -> {out_path.name} (ctrl_token='{ctrl_token}', ctrl_median={ctrl_median:.6g})")

    return out_paths


# Example: process a folder of CSVs (each CSV normalized to its own 6hr_Ctrl median)
normalize_csvs_to_ctrl_median_whole_file(
    "/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/96hr",
    value_col="Results",
    filename_col=0,   # first column
    ctrl_token="96hr_Ctrl",
    pseudocount=0.0,
)

✓ 96hr_Area (µm²).csv -> 96hr_Area (µm²)_Norm.csv (ctrl_token='96hr_Ctrl', ctrl_median=295.545)
✓ 96hr_Aspect Ratio.csv -> 96hr_Aspect Ratio_Norm.csv (ctrl_token='96hr_Ctrl', ctrl_median=1.42393)
✓ 96hr_Circularity.csv -> 96hr_Circularity_Norm.csv (ctrl_token='96hr_Ctrl', ctrl_median=0.629398)
✓ 96hr_Convex Hull Area (µm²).csv -> 96hr_Convex Hull Area (µm²)_Norm.csv (ctrl_token='96hr_Ctrl', ctrl_median=327.697)
✓ 96hr_Convex Hull Perimeter (µm).csv -> 96hr_Convex Hull Perimeter (µm)_Norm.csv (ctrl_token='96hr_Ctrl', ctrl_median=69.719)
✓ 96hr_Perimeter (µm).csv -> 96hr_Perimeter (µm)_Norm.csv (ctrl_token='96hr_Ctrl', ctrl_median=75.2366)
✓ 96hr_Roughness.csv -> 96hr_Roughness_Norm.csv (ctrl_token='96hr_Ctrl', ctrl_median=1.08025)


[PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/96hr/96hr_Area (µm²)_Norm.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/96hr/96hr_Aspect Ratio_Norm.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/96hr/96hr_Circularity_Norm.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/96hr/96hr_Convex Hull Area (µm²)_Norm.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/96hr/96hr_Convex Hull Perimeter (µm)_Norm.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/96hr/96hr_Perimeter (µm)_Norm.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/96hr/96hr_Roughnes

LPS ONLY, SHEET INPUT, ITERATES THROUGH ALL SHEETS

In [28]:
import os
from pathlib import Path
from typing import List, Optional, Union

import pandas as pd


def split_xlsx_sheets_to_files(
    input_excel: Union[str, os.PathLike],
    *,
    sheet_names: Optional[List[str]] = None,         # None => all sheets
    output_folder: Optional[Union[str, os.PathLike]] = None,
    output_format: str = "xlsx",                     # "xlsx" | "csv"
    include_sheet_name_in_filename: bool = True,
    overwrite: bool = False,
) -> List[Path]:
    """
    Split an Excel workbook into one file per sheet.

    Args:
        input_excel: Path to .xlsx workbook.
        sheet_names: List of sheets to export (None exports all).
        output_folder: Where to write outputs (default: <workbook_stem>_Split next to input).
        output_format: "xlsx" (one .xlsx per sheet) or "csv" (one .csv per sheet).
        include_sheet_name_in_filename: If True, output file name includes the sheet name.
        overwrite: If False, avoids overwriting by appending " (1)", " (2)", ...

    Returns:
        List of written output file paths.
    """
    input_excel = Path(input_excel)
    if not input_excel.exists() or not input_excel.is_file():
        raise ValueError(f"input_excel is not a file: {input_excel}")
    if input_excel.suffix.lower() not in {".xlsx", ".xlsm", ".xls"}:
        raise ValueError(f"input_excel must be an Excel file (.xlsx/.xlsm/.xls): {input_excel}")

    if output_format not in {"xlsx", "csv"}:
        raise ValueError("output_format must be 'xlsx' or 'csv'")

    xl = pd.ExcelFile(input_excel)
    sheets_to_export = sheet_names if sheet_names is not None else xl.sheet_names
    missing = [s for s in sheets_to_export if s not in xl.sheet_names]
    if missing:
        raise ValueError(f"Sheet(s) not found in workbook: {missing}")

    if output_folder is None:
        output_folder = input_excel.parent / f"{input_excel.stem}_Split"
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    def _fs_safe(name: str) -> str:
        name = str(name).strip()
        return "".join(c if c.isalnum() or c in ("-", "_") else "_" for c in name).strip("_") or "Sheet"

    def _avoid_overwrite(path: Path) -> Path:
        if overwrite or not path.exists():
            return path
        i = 1
        while True:
            candidate = path.with_name(f"{path.stem} ({i}){path.suffix}")
            if not candidate.exists():
                return candidate
            i += 1

    out_paths: List[Path] = []

    for sheet in sheets_to_export:
        df = pd.read_excel(xl, sheet_name=sheet)

        sheet_safe = _fs_safe(sheet)
        if include_sheet_name_in_filename:
            base_name = f"{input_excel.stem}_{sheet_safe}"
        else:
            base_name = sheet_safe

        if output_format == "csv":
            out_path = _avoid_overwrite(output_folder / f"{base_name}.csv")
            df.to_csv(out_path, index=False)
        else:
            out_path = _avoid_overwrite(output_folder / f"{base_name}.xlsx")
            # one-sheet workbook
            with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
                df.to_excel(writer, sheet_name=str(sheet)[:31], index=False)

        out_paths.append(out_path)
        print(f"✓ {input_excel.name}::{sheet} -> {out_path.name}")

    return out_paths


# Example:
split_xlsx_sheets_to_files(
    '/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time.xlsx',
    output_format="csv"
)

✓ Microglia Morphology by Incubation Time.xlsx::6hr -> Microglia Morphology by Incubation Time_6hr.csv
✓ Microglia Morphology by Incubation Time.xlsx::24hr -> Microglia Morphology by Incubation Time_24hr.csv
✓ Microglia Morphology by Incubation Time.xlsx::48hr -> Microglia Morphology by Incubation Time_48hr.csv
✓ Microglia Morphology by Incubation Time.xlsx::72hr -> Microglia Morphology by Incubation Time_72hr.csv
✓ Microglia Morphology by Incubation Time.xlsx::96hr -> Microglia Morphology by Incubation Time_96hr.csv


[PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/Microglia Morphology by Incubation Time_6hr.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/Microglia Morphology by Incubation Time_24hr.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/Microglia Morphology by Incubation Time_48hr.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/Microglia Morphology by Incubation Time_72hr.csv'),
 PosixPath('/Users/allisonpickle/Desktop/Normalized LPS Only/Microglia Morphology by Incubation Time_Split/Microglia Morphology by Incubation Time_96hr.csv')]